# TCSC

Threshold crossing sample count, from Arafat, Chowdhury and Hasan, "A simple time domain algorithm for the detection of ventricular fibrillation in electrocardiogram" (SIViP, 2011) [TCSC-2009]. This notebook implements it next to JEKOVA, with the same data access and scoring as `JEKOVA5.ipynb`.

## The idea

Normal rhythm spends most of its time near the isoelectric line: the P-R and S-T segments and the gaps between beats make up more than half of the signal. VF has no such quiet stretches. TCSC measures the percentage of time the ECG stays away from the baseline, relative to its own peak. It needs two operations, comparison and count.

TCSC improves on TCI (Thakor 1990), which counted threshold-crossing pulses in 1 s segments. It uses a longer window so that slow rhythms still contain a beat, a threshold on both polarities, and counts samples in place of pulses, so VF that stays under the threshold for a while loses less information.

## The algorithm

1. Preprocess the signal: subtract the mean, apply a moving average of order 5, a 1 Hz high-pass for drift, then a 30 Hz Butterworth low-pass (Amann's `filtering.m`).
2. Take a 3 s segment ($L_s = 3$) and multiply it by a cosine window that tapers the first and last 0.25 s and leaves the middle at 1:

$$
w(t) = \begin{cases}
\frac{1}{2}\left(1 - \cos(4 \pi t)\right) & 0 \le t \le \frac{1}{4} \\
1 & \frac{1}{4} \le t \le L_s - \frac{1}{4} \\
\frac{1}{2}\left(1 - \cos(4 \pi t)\right) & L_s - \frac{1}{4} \le t \le L_s
\end{cases}
$$

3. Divide the segment by its own absolute maximum, so the peak is 1.
4. Binarize: $b_i = 1$ when $|x_i| > V_0$, otherwise $b_i = 0$, with $V_0 = 0.2$.
5. Compute the percentage of samples above the threshold:

$$
N = \frac{\text{samples with } b_i = 1}{\text{samples in the segment}} \times 100
$$

6. Average over the six 3 s segments, 1 s apart, that fit in an 8 s episode ($L_e = 8$):

$$
N_a = \frac{1}{L_e - 2} \sum_{i=1}^{L_e - 2} N_i
$$

7. Call VF when $N_a > N_d$. The paper uses $N_d = 48$ for high specificity, and $25 < N_d < 35$ when sensitivity matters more.

## Parameters

| parameter | value | meaning |
|---|---:|---|
| $L_s$ | 3 s | analysis segment |
| taper | 0.25 s | cosine taper at each end of the segment |
| step | 1 s | distance between segments |
| $L_e$ | 8 s | decision episode, six segments |
| $V_0$ | 0.2 | threshold, as a fraction of the segment's peak |
| $N_d$ | 48 | decision threshold on $N_a$, in percent |

## Published results

The paper evaluates MITDB (both channels) and CUDB in 8 s episodes with a 1 s step, about 190,000 decisions per algorithm. An episode is VF when more than 4 s of it is annotated VF, and VT counts as no VF.

| database | sensitivity | specificity |
|---|---:|---:|
| MITDB | 97.48 | 99.33 |
| CUDB | 79.74 | 88.14 |
| overall | 80.97 | 98.51 |

The reported AUC is 98.44%, the best of the eight algorithms compared.

## What differs here

This is the published algorithm with the published constants, on the signal at 250 Hz (the paper's databases are at 360 Hz and 250 Hz). Data access and scoring follow `JEKOVA5.ipynb`: a window is shockable at 85% overlap with reference VF, where the paper uses more than 4 s of 8 s, and VT stays negative as in the paper. The filter runs causally, and the low-pass is order 5 as in the reference code, since the paper does not give the order.

In [1]:
# IMPORTS ###########################################

import importlib
import warnings
warnings.filterwarnings("ignore")

#####################################################

import os
import numpy as np
import pandas as pd
from scipy import signal
import matplotlib.pyplot as plt

from pxg import Stop
from pxg import FS
from pxg import EXG, Rids, Record
from pxg.rec import load_epi

from IPython.display import Markdown, display

%config InlineBackend.figure_format = "retina"

# show dataframes in full, no row/column truncation
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)

In [2]:
# TCSC algorithm

SEG = 3                 # L_s, analysis segment in seconds
EPI = 8                 # L_e, decision episode in seconds
V0 = 0.2                # threshold as a fraction of the segment peak
ND = 48                 # N_d, decision threshold in percent

HP_HZ = 1.0             # drift suppression
LP_HZ = 30.0            # Butterworth low-pass
LP_ORDER = 5

SPAN = EPI * FS         # the 8 s window, 2000 samples
FRAC = 0.85

# step 2: cosine taper of 0.25 s at each end, 1 in the middle
TAPER = signal.windows.tukey(SEG * FS, alpha = 0.5 / SEG)

# step 1: the paper's preprocessing (Amann's filtering.m)
def Filter(x) -> np.ndarray:
    y = np.asarray(x, dtype=float)
    y = y - np.mean(y)
    y = signal.lfilter(np.ones(5) / 5, [1.0], y)
    hp_b, hp_a = signal.butter(1, HP_HZ / (0.5 * FS), btype="highpass") # type: ignore
    y = signal.lfilter(hp_b, hp_a, y)
    lp_b, lp_a = signal.butter(LP_ORDER, LP_HZ / (0.5 * FS), btype="lowpass") # type: ignore
    return signal.lfilter(lp_b, lp_a, y)
pass #def

# steps 2-6: N for every 3 s segment 1 s apart, then the average over 8 s
def TCSC(y: np.ndarray) -> np.ndarray:
    segs = np.lib.stride_tricks.sliding_window_view(y, SEG * FS)[::FS]
    x = np.abs(segs * TAPER)
    peak = x.max(axis=1, keepdims=True)
    N = 100.0 * (x > V0 * peak).mean(axis=1)
    return np.convolve(N, np.ones(EPI - SEG + 1) / (EPI - SEG + 1), "valid")
pass #def

def Process(rec: Record) -> tuple[pd.DataFrame, np.ndarray]:
    insig = rec.Sensor

    vfib = np.zeros(len(insig), dtype=int)
    for epi in rec.RefEpi:
        if epi.Name in ["#VT", "VFL", "VF", "WF", "VFIB"]:
            vfib[epi.Time:epi.End+1] = 1
        pass #if
    pass #for

    tcsc = TCSC(Filter(insig))
    secs = len(tcsc)
    total = np.r_[0, np.cumsum(vfib)]
    pon = np.arange(secs) * FS
    pof = pon + SPAN

    df = pd.DataFrame(dict(db=rec.DB, rid=rec.RID, pon=pon, pof=pof, vfb=total[pof] - total[pon], tcsc=tcsc))
    return df, vfib
pass #def

def Prepare(db: str, use_wfon = False) -> pd.DataFrame:
    os.makedirs(f"../work/tcsc/{db}", exist_ok=True)

    rows = []
    for rid in Rids(db):
        rec = Record(db, rid, wfon=use_wfon)
        df, vfb = Process(rec)
        rows.append(df)

        # save the per-record VFB mask as a compact binary (uint8)
        np.save(f"../work/tcsc/{db}/{rid}.ref.npy", vfb.astype(np.uint8))
    pass #for

    df = pd.concat(rows, ignore_index=True)
    df.to_csv(f"../work/tcsc/{db}_df{EPI}.tsv", index=False, sep="\t", float_format="%.3f")
    return df
pass #def

def Classify(wf: pd.DataFrame, nd: float = ND) -> np.ndarray:
    return np.where(wf["tcsc"].to_numpy(dtype=float) > nd, "S", "N")
pass #def

In [3]:
# Evaluate 

def Rates(tp: int, fn: int, fp: int, tn: int) -> dict:
    """Confusion counts and derived rates, shared by the window and sample tables."""
    return dict(
        TN=tn, TP=tp, FN=fn, FP=fp, ERR = fp + fn,
        SPC = round(100 * tn / (tn + fp), 2) if tn + fp else float("nan"),
        SEN = round(100 * tp / (tp + fn), 2) if tp + fn else float("nan"),
        PPV = round(100 * tp / (tp + fp), 2) if tp + fp else float("nan"),
        F1  = round(100 * 2 * tp / (2 * tp + fp + fn), 2) if 2 * tp + fp + fn else float("nan"),
    )
pass #def

def MetricsWin(call, shock, nonsh, mixed) -> dict:
    """Window-level confusion for one boolean slice of a database."""
    tp = int(((call == "S") & shock).sum())
    fn = int(((call != "S") & shock).sum())
    fp = int(((call == "S") & nonsh).sum())
    tn = int(((call != "S") & nonsh).sum())
    return dict(nonsh=int((nonsh & ~mixed).sum()), mixed=int(mixed.sum()), shock=int(shock.sum()), **Rates(tp, fn, fp, tn))
pass #def

def Evaluate(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Per-record SAMPLE rows (same columns as EvaluateWin) plus a TOTAL row.

    The window decisions are mapped back to a per-sample prediction mask (a sample
    is shockable if any covering window was called "S"), compared against the
    per-sample reference, and the mask saved as {db}/{rid}.vfb.npy.
    """

    call = Classify(df)

    # frac is a cut, not a band: every window lands in exactly one class.
    shock = (df["vfb"] >= FRAC * SPAN).to_numpy()
    nonsh = ~shock
    # windows straddling an episode edge: some VF present but under the cut,
    # scored as negatives, reported apart from the clear negatives
    mixed = (df["vfb"].to_numpy() > 0) & nonsh

    db  = df["db"].iloc[0]
    rid = df["rid"].to_numpy()
    pon = df["pon"].to_numpy()
    pof = df["pof"].to_numpy()

    wins = []
    samp = []
    stot = [0, 0, 0, 0]                    # pooled sample TP, FN, FP, TN for TOTAL
    for r in pd.unique(rid):
        m = rid == r
        wins.append({"db": db, "rid": str(r), **MetricsWin(call[m], shock[m], nonsh[m], mixed[m])})

        ref  = np.load(f"../work/tcsc/{db}/{r}.ref.npy").astype(bool)
        pred = np.zeros(len(ref), dtype=bool)
        for a, b, cl in zip(pon[m], pof[m], call[m]):
            if cl == "S":
                pred[a:b] = True
            pass #if
        pass #for
        np.save(f"../work/tcsc/{db}/{r}.vfb.npy", pred.astype(np.uint8))

        # Save as detected annotations
        os.makedirs(f"../work/tcsc/anv/{db}", exist_ok=True)
        with open(f"../work/tcsc/anv/{db}/{r}.anv", "w") as f:
            # write only the samples where the mask changes
            for ix in np.r_[0, np.flatnonzero(pred[1:] != pred[:-1]) + 1].tolist():
                f.write(f"{ix},+,{'(WF' if pred[ix] else '(NF'}\n")
            pass #for
            f.write(f"{len(pred)},+,(END\n")
        pass #with

        tp = int((pred & ref).sum());  fn = int((~pred & ref).sum())
        fp = int((pred & ~ref).sum()); tn = int((~pred & ~ref).sum())
        for i, v in enumerate((tp, fn, fp, tn)):
            stot[i] += v
        pass #for
        samp.append({"db": db, "rid": str(r), "nonsh": tn + fp, "mixed": 0, "shock": tp + fn, **Rates(tp, fn, fp, tn)})
    pass #for

    wins.sort(key=lambda x: x["ERR"], reverse=True)   # sort by error count, descending

    wins.append({"db": db, "rid": "TOTAL", **MetricsWin(call, shock, nonsh, mixed)})
    tp, fn, fp, tn = stot
    samp.append({"db": db, "rid": "TOTAL", "nonsh": tn + fp, "mixed": 0, "shock": tp + fn, **Rates(tp, fn, fp, tn)})

    return pd.DataFrame(wins), pd.DataFrame(samp)
pass #def

In [4]:
# PREPARE + EVAL EXEC

USE_WFON = False
DATABASES = ["mitdb", "ahadb", "cudb", "edb"]

wins, samp = [], []
for db in DATABASES:
    print(db)
    df = Prepare(db, USE_WFON)
    rw, rs = Evaluate(df)
    wins.append(rw[rw["rid"] == "TOTAL"])
    samp.append(rs[rs["rid"] == "TOTAL"])
pass #for

display(Markdown("## Summary, per sample"))
display(pd.concat(samp, ignore_index=True))

display(Markdown("## Summary, per window"))
display(pd.concat(wins, ignore_index=True))

mitdb


ahadb


cudb


edb


## Summary, per sample

,db,rid,nonsh,mixed,shock,TN,TP,FN,FP,ERR,SPC,SEN,PPV,F1
0,mitdb,TOTAL,21590914,0,35966,21187130,35966,0,403784,403784,98.13,100.00,8.18,15.12
1,ahadb,TOTAL,33915108,0,1355548,32806517,1290159,65389,1108591,1173980,96.73,95.18,53.78,68.73
2,cudb,TOTAL,3495017,0,949143,2728857,811090,138053,766160,904213,78.08,85.45,51.42,64.21
3,edb,TOTAL,161832960,0,0,159510210,0,0,2322750,2322750,98.56,NaN,0.00,0.00


## Summary, per window

,db,rid,nonsh,mixed,shock,TN,TP,FN,FP,ERR,SPC,SEN,PPV,F1
0,mitdb,TOTAL,85978,70,112,85132,107,5,916,921,98.94,95.54,10.46,18.85
1,ahadb,TOTAL,135011,98,5353,133106,4569,784,2003,2787,98.52,85.35,69.52,76.63
2,cudb,TOTAL,13453,540,3507,11847,2590,917,2146,3063,84.66,73.85,54.69,62.84
3,edb,TOTAL,646650,0,0,639666,0,0,6984,6984,98.92,NaN,0.00,0.00
